# BorderMesh Tamper CNN — GPU Training Notebook

**Before running:** Runtime menu -> Change runtime type -> GPU (T4 is fine).

## Why this notebook exists

Every local training attempt tonight (on a CPU-only laptop, ~25k-parameter model) hit the same wall: three independent variables were each tested in isolation --
- **Bigger model** (25k -> ~250k params): overfit, worse on fresh genuine specimens
- **More regularization + augmentation on top of that**: also worse (train/inference distribution mismatch)
- **5x more of our own training data, same small model**: also didn't generalize

All three point the same direction: this isn't a tuning problem solvable with one more local run. It needs real GPU-backed training on a properly-sized model with much more real, diverse data than a CPU laptop can practically train on. That's what this notebook does, using your Google Drive for storage.

## What this notebook does, in order
1. Mounts your Google Drive and clones the BorderMesh repo (so we reuse the exact same generator/loader code already proven correct, not a reimplementation)
2. Downloads IDNet (CC0-licensed, ~837k synthetic ID document images with fraud variants + JSON metadata) into your Drive -- pick how many parts based on how much of your 5TB you want to commit
3. **Inspects the real IDNet file structure before we build a parser for it** -- the paper describing IDNet doesn't give exact JSON key names, so rather than guess, we look at a real file first
4. Blends: our own synthetic generator (unlimited, free) + CASIA v2.0 (real splices, needs your Kaggle token) + IDNet (real synthetic ID fraud, needs the inspection step above)
5. Trains a proper transfer-learning model (MobileNetV3-Small, ImageNet-pretrained) -- GPU removes the reason the local model had to stay tiny
6. Runs the exact same sanity check used all night: generate fresh, never-seen genuine specimens and confirm the model doesn't confidently misflag them, before you trust the result
7. Exports the trained weights to your Drive for download and integration back into the project

**Stop and check with me before step 7** if the sanity check in step 6 doesn't look clean -- paste me the printed numbers and I'll help interpret them, the same way we did for every local attempt tonight.

## 1. Mount Drive, clone the repo, install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Everything downloaded/exported lives under this Drive folder so it survives
# across Colab sessions (Colab's local disk is wiped when the runtime recycles).
WORK_DIR = '/content/drive/MyDrive/bordermesh_tamper_training'
import os
os.makedirs(WORK_DIR, exist_ok=True)
print('Working directory:', WORK_DIR)

In [ ]:
# Replace with your fork/repo URL if different.
REPO_URL = 'https://github.com/ArihantK15/SyntaxSquad.git'
REPO_DIR = '/content/SyntaxSquad'

!git clone {REPO_URL} {REPO_DIR}

In [ ]:
!pip install -q torch torchvision opencv-python-headless pillow

# Install the exact same font the production Docker image uses
# (backend/Dockerfile: fonts-dejavu-core). Without this, SyntheticDocumentGenerator
# falls back to PIL's tiny default bitmap font on Colab's Ubuntu image, which
# renders documents differently than production -- the exact train/inference
# font mismatch that was root-caused and fixed earlier tonight (see
# scripts/train_tamper_cnn.py's TAMPER_TRAIN_FONT handling).
!apt-get -qq install -y fonts-dejavu-core > /dev/null

import sys
sys.path.insert(0, f'{REPO_DIR}/backend')
sys.path.insert(0, f'{REPO_DIR}/scripts')

from app.utils.synthetic_generator import SyntheticDocumentGenerator
_PRODUCTION_FONT = '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
import os as _os
assert _os.path.exists(_PRODUCTION_FONT), 'DejaVu font install failed -- check the apt-get output above'
SyntheticDocumentGenerator._FONT_CANDIDATES = [_PRODUCTION_FONT]
print('Using production font:', _PRODUCTION_FONT)

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go to Runtime > Change runtime type > GPU)')

## 2. Download IDNet into Drive

IDNet is split into 8 parts on Zenodo, ~490GB total. **You don't need all of it.** Pick a few country/state parts below -- more parts = more diversity, but also more download time and Drive space. A good starting point is 3-4 parts (~40-60GB) covering a mix of countries.

Zenodo doesn't support resuming partial downloads well, so this can take a while depending on Colab's network speed to Zenodo (usually faster than a home connection, but not guaranteed). Each `!wget` below writes straight into your mounted Drive -- if the runtime disconnects, already-downloaded files aren't lost.

Record URLs (from the IDNet paper, Zenodo): use `action: "list"` in your own browsing to confirm current links if these ever move -- as of this notebook, part 2 (record 10570393) covers Azerbaijan, Serbia, Greece, Finland.

In [ ]:
IDNET_DIR = f'{WORK_DIR}/idnet_raw'
os.makedirs(IDNET_DIR, exist_ok=True)

# Start with the smallest single file to validate the pipeline end-to-end
# before committing to a bigger download. Add more URLs here once step 3
# below confirms the parser works.
IDNET_URLS = [
    'https://zenodo.org/records/10570393/files/fin.zip',   # Finland, ~4.4GB
    # 'https://zenodo.org/records/10570393/files/grc.zip',  # Greece, ~6.4GB
    # 'https://zenodo.org/records/10570393/files/aze.zip',  # Azerbaijan, ~13.0GB
    # 'https://zenodo.org/records/10570393/files/srb.zip',  # Serbia, ~24.9GB
]

for url in IDNET_URLS:
    fname = url.split('/')[-1]
    dest = f'{IDNET_DIR}/{fname}'
    if os.path.exists(dest):
        print(f'{fname} already downloaded, skipping.')
        continue
    print(f'Downloading {fname} ...')
    !wget -q --show-progress -O {dest} {url}

In [ ]:
# Unzip just the first downloaded part for now -- don't extract everything
# until step 3 confirms we're parsing it correctly.
import zipfile

first_zip = f'{IDNET_DIR}/{IDNET_URLS[0].split("/")[-1]}'
EXTRACT_DIR = f'{IDNET_DIR}/extracted'
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(first_zip) as zf:
    names = zf.namelist()
    print(f'{len(names)} entries in {first_zip}')
    print('First 20 entries:')
    for n in names[:20]:
        print(' ', n)

## 3. STOP AND LOOK: inspect the real structure before writing a parser

Run the cell above first. The output tells us the actual folder layout (e.g. is it `fin/authentic/...` and `fin/fraud/...`, or something else?) and the actual metadata filenames. Run the cell below to print one real metadata JSON file in full -- **paste both outputs back to Claude** so the loader in step 4 is written against the real schema instead of a guess from the paper's prose description.

In [ ]:
import json

# Find and print the first .json file we can locate, so we can see the real
# metadata schema (field names for fraud type, tampered field, etc.).
with zipfile.ZipFile(first_zip) as zf:
    json_names = [n for n in zf.namelist() if n.lower().endswith('.json')]
    print(f'{len(json_names)} JSON files found. Showing the first one:\n')
    if json_names:
        with zf.open(json_names[0]) as f:
            data = json.load(f)
            print(json.dumps(data, indent=2))
    else:
        print('No JSON files found at the top level -- check the entry list above for where metadata actually lives (it may be nested, or a single index file rather than per-document).')

## 4. Build the combined training set

This blends three sources using the exact same domain-balanced sampling approach used in every local attempt tonight (`scripts/train_tamper_cnn.py`) -- each domain gets roughly equal training exposure regardless of its raw size, so a large real dataset can't drown out our own generator's style (the failure mode that broke the very first CASIA-only local attempt).

**The IDNet loading function below is a best-effort first pass based on the paper's description, not the confirmed real schema.** Once you've pasted me the output from step 3, I'll rewrite `load_idnet_patches` to match exactly -- treat this cell as a draft, expect it to need a fix.

In [ ]:
import random
import numpy as np
import cv2
from pathlib import Path
from typing import Tuple, Optional

from generate_tamper_training_data import generate_dataset, load_casia_patches, _random_patch, _patch_centered_at, PATCH_SIZE

def load_idnet_patches(
    extract_dir: str,
    patches_per_authentic: int = 4,
    patches_per_fraud: int = 4,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    DRAFT -- adjust field/path names below once step 3's real output is known.

    Expected (per the IDNet paper, not yet confirmed): each document has one
    authentic image + up to 4 fraud variants (face morphing, portrait
    substitution, text-field replacement, mixed), plus a JSON metadata file
    per document naming the fraud type and, for portrait-related fraud, the
    face image identifiers involved (which may let us localize a face-region
    bounding box even without exact pixel coordinates for text edits).
    """
    root = Path(extract_dir)
    image_paths = list(root.rglob('*.jpg')) + list(root.rglob('*.png'))
    json_paths = {p.stem: p for p in root.rglob('*.json')}

    patches, labels = [], []
    resolved, unresolved = 0, 0

    for img_path in image_paths:
        meta_path = json_paths.get(img_path.stem)
        is_fraud = False
        if meta_path is not None:
            try:
                meta = json.loads(meta_path.read_text())
                is_fraud = bool(meta.get('fraud_type') or meta.get('is_fraud') or meta.get('fraud'))
            except Exception:
                pass
        else:
            # Fall back to path-based heuristic if no matching JSON -- adjust
            # once real folder names are known from step 3.
            is_fraud = 'fraud' in str(img_path).lower() or 'forg' in str(img_path).lower()

        img = cv2.imread(str(img_path))
        if img is None:
            continue

        n = patches_per_fraud if is_fraud else patches_per_authentic
        for _ in range(n):
            patches.append(_random_patch(img))
            labels.append(1 if is_fraud else 0)
        if is_fraud:
            resolved += 1
        else:
            unresolved += 1

    print(f'IDNet: {len(image_paths)} images found, {resolved} flagged fraud, {unresolved} flagged authentic')
    if not patches:
        raise RuntimeError('No IDNet patches extracted -- the folder/JSON structure assumed above does not match reality. Share the step 3 output with Claude.')
    return np.stack(patches).astype(np.uint8), np.array(labels, dtype=np.int64)

In [ ]:
# --- Kaggle credentials for CASIA v2.0 (same dataset used locally) ---
# Upload your kaggle.json (Kaggle account -> Settings -> Create New Token)
# using the file upload icon in Colab's left sidebar, then point this at it.
import shutil
os.makedirs('/root/.kaggle', exist_ok=True)
if os.path.exists('/content/kaggle.json'):
    shutil.copy('/content/kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    !pip install -q kaggle
    CASIA_DIR = f'{WORK_DIR}/casia2'
    if not os.path.exists(CASIA_DIR):
        os.makedirs(CASIA_DIR, exist_ok=True)
        !kaggle datasets download -d divg07/casia-20-image-tampering-detection-dataset -p {CASIA_DIR} --unzip
else:
    print('No kaggle.json uploaded -- skipping CASIA, training will use own-synthetic + IDNet only.')
    CASIA_DIR = None

In [ ]:
print('Generating our own synthetic documents (free, unlimited)...')
own_patches, own_labels = generate_dataset(n_docs=1500, patches_per_doc=8)
print(f'  {len(own_patches)} own-synthetic patches')

domain = np.zeros(len(own_patches), dtype=np.int64)  # 0 = own-synthetic
all_patches, all_labels = own_patches, own_labels

if CASIA_DIR:
    print('Loading CASIA v2.0...')
    casia_patches, casia_labels = load_casia_patches(CASIA_DIR)
    print(f'  {len(casia_patches)} CASIA patches')
    all_patches = np.concatenate([all_patches, casia_patches], axis=0)
    all_labels = np.concatenate([all_labels, casia_labels], axis=0)
    domain = np.concatenate([domain, np.ones(len(casia_patches), dtype=np.int64)])  # 1 = CASIA

print('Loading IDNet...')
idnet_patches, idnet_labels = load_idnet_patches(EXTRACT_DIR)
print(f'  {len(idnet_patches)} IDNet patches')
all_patches = np.concatenate([all_patches, idnet_patches], axis=0)
all_labels = np.concatenate([all_labels, idnet_labels], axis=0)
domain = np.concatenate([domain, np.full(len(idnet_patches), 2, dtype=np.int64)])  # 2 = IDNet

print(f'\nCombined total: {len(all_patches)} patches '
      f'({int((domain==0).sum())} own-synthetic, {int((domain==1).sum())} CASIA, {int((domain==2).sum())} IDNet)')

## 5. Train a transfer-learning model on the GPU

MobileNetV3-Small, ImageNet-pretrained, fine-tuned on our patches. This is only viable with a GPU -- it's the model class we deliberately avoided locally because it would have been far too slow on CPU. Domain-balanced sampling and per-domain validation tracking follow the same pattern as every local run tonight, for the same reason: a combined metric can look good while hiding a regression on the domain that actually matters (our own generator's style).

In [ ]:
# NOTE: to_tensor_dataset below materializes the whole dataset in memory
# at once. Fine for the single Finland part (~tens of thousands of patches);
# if you add more IDNet parts and this cell runs out of RAM, that's the fix
# needed -- switch to a Dataset that resizes patches lazily per-batch
# instead of upfront, and flag it to Claude rather than guessing at a fix.
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split, WeightedRandomSampler
import torchvision.models as models
import torchvision.transforms.v2 as T

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PATCH_INPUT_SIZE = 96  # upsampled from the native 64x64 patch for the pretrained backbone

def to_tensor_dataset(patches, labels):
    x = torch.from_numpy(patches).permute(0, 3, 1, 2).float() / 255.0
    x = torch.nn.functional.interpolate(x, size=(PATCH_INPUT_SIZE, PATCH_INPUT_SIZE), mode='bilinear', align_corners=False)
    y = torch.from_numpy(labels).long()
    return TensorDataset(x, y)

dataset = to_tensor_dataset(all_patches, all_labels)
n_val = int(len(dataset) * 0.2)
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42))

train_domain = domain[train_ds.indices]
domain_counts = np.bincount(train_domain, minlength=3)
per_sample_weight = 1.0 / np.maximum(domain_counts[train_domain], 1)
sampler = WeightedRandomSampler(per_sample_weight, num_samples=n_train, replacement=True)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()
EPOCHS = 15

In [ ]:
def domain_val_accuracy(model, val_ds, domain, which):
    idxs = [i for i in val_ds.indices if domain[i] == which]
    if not idxs:
        return None
    xb = torch.stack([val_ds.dataset[i][0] for i in idxs]).to(device)
    yb = torch.stack([val_ds.dataset[i][1] for i in idxs]).to(device)
    model.eval()
    with torch.no_grad():
        preds = model(xb).argmax(dim=1)
    return (preds == yb).float().mean().item()

best_own_acc = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    own_acc = domain_val_accuracy(model, val_ds, domain, 0)
    casia_acc = domain_val_accuracy(model, val_ds, domain, 1)
    idnet_acc = domain_val_accuracy(model, val_ds, domain, 2)
    fmt = lambda a: f'{a:.3%}' if a is not None else 'n/a'

    marker = ''
    selection_metric = own_acc if own_acc is not None else 0.0
    if selection_metric > best_own_acc:
        best_own_acc = selection_metric
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        marker = '  (best so far)'

    print(f'epoch {epoch:2d}/{EPOCHS}  loss={total_loss/n_train:.4f}  '
          f'own={fmt(own_acc)}  casia={fmt(casia_acc)}  idnet={fmt(idnet_acc)}{marker}')

model.load_state_dict(best_state)
print(f'\nBest own-domain validation accuracy: {best_own_acc:.3%}')

## 6. Sanity check -- the gate that caught every bad checkpoint tonight

Same discipline as every local attempt: generate specimens the model has never seen, and check its real-world behavior directly, not just a held-out validation number. **Do not treat a good validation accuracy above as success on its own -- every failed local attempt tonight also had a good validation number.**

In [ ]:
from app.utils.synthetic_generator import SyntheticDocumentGenerator
import tempfile

def patch_tamper_prob(model, img_bgr, y1, y2, x1, x2):
    patch = img_bgr[max(0,y1):min(img_bgr.shape[0],y2), max(0,x1):min(img_bgr.shape[1],x2)]
    patch_rgb = cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)
    patch_resized = cv2.resize(patch_rgb, (PATCH_INPUT_SIZE, PATCH_INPUT_SIZE))
    t = torch.tensor(patch_resized, dtype=torch.float32).permute(2,0,1).unsqueeze(0).to(device) / 255.0
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(t), dim=1).squeeze().cpu().numpy()
    return float(probs[1])

print('Sanity-checking against 5 FRESH genuine specimens (never seen during training)...')
risks = []
for i in range(5):
    out_path = tempfile.mktemp(suffix='.jpg')
    SyntheticDocumentGenerator.generate_document(out_path=out_path, mode='genuine', doc_number=f'S{9000000+i}')
    img = cv2.imread(out_path)
    h, w = img.shape[:2]
    regions = [
        ('portrait', (int(h*0.15), int(h*0.75), int(w*0.05), int(w*0.45))),
        ('center', (h//2-64, h//2+64, w//2-64, w//2+64)),
        ('mrz', (int(h*0.75), h, 0, w)),
    ]
    region_probs = {name: patch_tamper_prob(model, img, *coords) for name, coords in regions}
    risks.append(max(region_probs.values()))
    print(f'  specimen {i}: {region_probs}')
    os.remove(out_path)

mean_risk = sum(risks) / len(risks)
print(f'\nMax-region tamper probability across 5 fresh genuine specimens: {risks} (mean {mean_risk:.2f})')
if mean_risk >= 0.5:
    print('WARNING: this is not looking clean. Paste this output to Claude before exporting -- do not treat this as a working checkpoint yet.')
else:
    print('Looks clean. Also manually check a real forgery (mode="photo_replaced") scores HIGH before exporting.')

## 7. Export weights (only after the sanity check above looks clean)

This exports to Drive. Download the file from Drive and hand it back for integration -- note that `tamper_service.py` currently extracts 64x64 patches and loads a state dict shaped for `LightweightForensicCNN`; integrating this MobileNetV3-based checkpoint means updating both the patch extraction size (to match `PATCH_INPUT_SIZE` above) and the model class it loads, not just dropping the file in. Flag this to Claude when you're ready to integrate rather than doing it yourself -- it touches production inference code.

In [ ]:
EXPORT_PATH = f'{WORK_DIR}/tamper_cnn_mobilenetv3.pth'
torch.save(model.state_dict(), EXPORT_PATH)
print(f'Saved to {EXPORT_PATH}')
print('Download this file from Drive and share it (or the Drive link) for integration.')